# Model Training

Prepare the Training Dataset and experiment with different models for automatically predict the `Test Results` based on a list of patient's features.

# Setup Notebook

## Imports

In [1]:
# Import Standard Libraries
import os
import mlflow
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Import Package Modules
from src.general_utils.general_utils import read_configuration
from src.data_preparation.data_preparation import HealthcareDataPreparation
from src.model_training.model_training import ModelTrainer

## Define Configurations

In [2]:
# Retrieve root path
root_path = Path(os.getcwd()).parents[1]

In [3]:
# Read configuration variables
config = read_configuration(root_path / 
                            'configuration' / 
                            'healthcare_classification_config.yaml')

[05/18/2024 17:04:47 - general_utils] INFO - read_configuration - Start
[05/18/2024 17:04:47 - general_utils] INFO - read_configuration - Reading /Users/s.porreca/Projects/MediBioticsAI/configuration/healthcare_classification_config.yaml
[05/18/2024 17:04:47 - general_utils] INFO - read_configuration - Configuration file /Users/s.porreca/Projects/MediBioticsAI/configuration/healthcare_classification_config.yaml read successfully
[05/18/2024 17:04:47 - general_utils] INFO - read_configuration - End


In [4]:
# Extract configuration variables
# -------- Data Pipeline -------
dataset_config = config['healthcare_data_pipeline_config']['dataset']
data_transformations_config = config['healthcare_data_pipeline_config']['data_transformations']
features_config = config['healthcare_data_pipeline_config']['features']
labels_config = config['healthcare_data_pipeline_config']['labels']
train_test_split_config = config['healthcare_data_pipeline_config']['train_test_split']
# ------------------------------

# -------- Model Training -------
model_training_config = config['model_training_config']
# -------------------------------

# Read Data

## Healthcare Dataset

In [5]:
# Define path
data_path = (root_path / 
                        dataset_config['data_path'][0] / 
                        dataset_config['data_path'][1] / 
                        dataset_config['data_path'][2])

In [6]:
# Read data
dataset = pd.read_csv(data_path, 
                                 parse_dates=dataset_config['date_columns'])

# Data Pipeline

## Define Data Preparation Pipeline

In [7]:
# Instance the data preparation pipeline object
data_preparation = HealthcareDataPreparation(data_transformations_config,
                                             features_config, 
                                             labels_config)

[05/18/2024 17:04:47 - HealthcareDataPreparation] INFO - __init__ - Initialise object attributes


In [8]:
# Get the training data preparation pipeline
data_preparation_pipeline = data_preparation.build_training_data_preparation_pipeline()

[05/18/2024 17:04:47 - HealthcareDataPreparation] INFO - build_training_data_preparation_pipeline - Start
[05/18/2024 17:04:47 - HealthcareDataPreparation] INFO - build_training_data_preparation_pipeline - Build the Numerical Data Pipeline
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Start
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Building steps
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Skipping Feature Engineering step
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Adding SimpleImputer Imputation step
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Adding MinMaxScaler Standardisation step
[05/18/2024 17:04:47 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Skipping Normalization step
[05/18/2024 17:04:47 - data_preparation_utils] 

In [9]:
data_preparation_pipeline

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(copy=False,
                                                                strategy='median')),
                                                 ('standardisation',
                                                  MinMaxScaler())]),
                                 ['Age', 'Billing Amount', 'Room Number']),
                                ('categorical',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(copy=False,
                                                                fill_value='unknown',
                                                                strategy='constant')),
                                                 ('one_hot_encoding',
                                                  OneHotEncoder())]),
                                 ['Gender', 'Blood Type', 'Medical Condition',
                                  'Insurance Provider', 'Admission Type',
                                  'Medication'])])

## Train & Test Split

In [10]:
# Define the features to include
features = features_config['numerical'] + \
           features_config['categorical']

# Define the labels to include
labels = labels_config

print(f'Features: {features}')
print()
print(f'Labels: {labels}')

Features: ['Age', 'Billing Amount', 'Room Number', 'Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Admission Type', 'Medication']

Labels: ['Test Results']


In [11]:
# Define X and y for the training set
X = dataset[features]
y = dataset[labels]

In [12]:
# Split training data into train and validation
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=train_test_split_config['test_size'], 
                                                    random_state=train_test_split_config['random_state'])

# Model Training

## Setup Training

In [13]:
# Set MLflow Experiment
mlflow_experiment_name = model_training_config['mlflow']['experiment_name']

# Set MLflow Experiment
mlflow.set_experiment(mlflow_experiment_name)

<Experiment: artifact_location='file:///Users/s.porreca/Projects/MediBioticsAI/notebooks/healthcare_classification/mlruns/388746619317475161', creation_time=1714156779505, experiment_id='388746619317475161', last_update_time=1714156779505, lifecycle_stage='active', name='Version 1.0.x', tags={}>

In [14]:
# Initialise trained models dictionary
models = {}

# Initialize DataFrame of models performance
performance = pd.DataFrame(columns=model_training_config['metrics'])

## Logistic Regression

In [15]:
# Define the model
model_lr = LogisticRegression()

# Create a ModelTrainer
model_trainer_lr = ModelTrainer(model_name=model_training_config['linear_regression']['model_name'], 
                                model=model_lr, 
                                data_pipeline=data_preparation_pipeline)

[05/18/2024 17:04:47 - ModelTrainer] INFO - __init__ - Initialise object attributes


In [16]:
model_trainer_lr.pipeline

In [17]:
# Start an MLflow run
with mlflow.start_run(run_name=model_training_config['linear_regression']['mlflow_run_name']):

    # Fit the model trainer
    model_trainer_lr.bundle_and_fit_pipeline(X_train, y_train)
    
    # Evaluate the model trainer
    evaluation = model_trainer_lr.evaluate_pipeline(X_test, y_test, model_training_config['metrics'])
    
    # Log model's evaluation metrics
    mlflow.log_metrics(evaluation.to_dict()['Value'])
    
    # Log model's features
    mlflow.log_params({'Features': features, 
                       'Labels': labels,
                       'Data Transformations': data_transformations_config,
                       'Model Initial Hyperparameters': None,
                       'Model Optimised Hyperparameters': None})
    
    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model_trainer_lr.pipeline,
        artifact_path='model_artifacts',
        input_example=X_train.head(1),
        registered_model_name=model_training_config['linear_regression']['mlflow_run_name']
    )

[05/18/2024 17:04:47 - ModelTrainer] INFO - bundle_and_fit_pipeline - Start
[05/18/2024 17:04:47 - ModelTrainer] INFO - bundle_and_fit_pipeline - Bundle the pipeline
[05/18/2024 17:04:47 - ModelTrainer] INFO - bundle_and_fit_pipeline - Fit the pipeline
[05/18/2024 17:04:47 - ModelTrainer] INFO - bundle_and_fit_pipeline - End
[05/18/2024 17:04:47 - ModelTrainer] INFO - evaluate_pipeline - Start
[05/18/2024 17:04:47 - ModelTrainer] INFO - evaluate_pipeline - Compute predictions
[05/18/2024 17:04:47 - ModelTrainer] INFO - evaluate_pipeline - Evaluate pipeline
[05/18/2024 17:04:47 - model_training_utils] INFO - compute_multi_classification_metrics - Start


/Users/s.porreca/.local/share/virtualenvs/cheat_sheets-EiW5VkhA/lib/python3.10/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


ValueError: could not convert string to float: 'Inconclusive'